In [ ]:
import os
import scanpy as sc
import sctm
import anndata as ad
import pandas as pd
import squidpy as sq
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [ ]:
in_dir = "../../results/06_spahdmap/05_prepare_data"
out_dir = "../../results/07_scTM/01_run_scTM"

os.makedirs(out_dir, exist_ok=True)

In [ ]:
sample_list = ['15837', '15852_1', '15852_2', '16512_1', '16512_2', '16512_3',
               '21022_1', '21022_2', '8262_1', '8262_2',
               'B23-03794-5', 'B23-05785-7-1', 'B23-05785-7-2',
               'X1842_1', 'X1842_2', 'X1845', 'X1899_1', 'X1899_2', 'X1899_3', 'X1912_1', 'X1912_2']

# sample_list = ['15837', '15852_1', '15852_2', 'B23-03794-5', 'B23-05785-7-2', 'X1842_1']
# sample_list = ['15837', '15852_1']

In [ ]:
adata_list = []
for sample in sample_list:
    adata = sc.read_h5ad(f"{in_dir}/{sample}.h5ad")
    adata.obs['library_id'] = sample
    adata_list.append(adata)

adata = ad.concat(adata_list)

In [ ]:
adata = adata[:, ~adata.var_names.str.startswith(("MT-", 'RPL', 'RPS'))]

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sctm.pp.filter_genes(adata, min_cutoff=0.01, max_cutoff=0.95)

In [ ]:
sctm.pp.batch_highly_variable_genes(adata, 
                                    n_top_genes=3000, 
                                    layer="counts", 
                                    subset=False, 
                                    batch_key="library_id")

In [ ]:
sq.gr.spatial_neighbors(adata, library_key="library_id", n_rings=1)

In [ ]:
sctm.seed.seed_everything(5)
model = sctm.stamp.STAMP(
    adata[:, adata.var.highly_variable],
    n_topics=15,
    layer = "counts",
    mode="sign",
    dropout=0.2,
    categorical_covariate_keys = ["library_id"],
    continous_covariate_keys=['total_counts'],
    gene_likelihood = "nb")

In [ ]:
model.train(learning_rate = 3e-04, min_epochs = 200)

In [ ]:
topic_prop = model.get_cell_by_topic()
beta = model.get_feature_by_topic(pseudocount = 0.0)
for i in topic_prop.columns:
    adata.obs[i] = topic_prop[i]

In [ ]:
# adata.write_h5ad(f"{out_dir}/scTM_15topics.h5ad")

In [ ]:
df_beta = pd.DataFrame(beta)
df_topic = pd.DataFrame(topic_prop)

In [ ]:
df_beta.to_csv(f"{out_dir}/scTM_15topics_beta.csv")
df_topic.to_csv(f"{out_dir}/scTM_15topics_topic_proportions.csv")

In [ ]:
adata.obsm["X_sctm"] = topic_prop

In [ ]:
sc.pp.neighbors(adata, use_rep="X_sctm")
sc.tl.umap(adata)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sc.pl.umap(adata, color='library_id', show=False, ax=ax, frameon=False, legend_loc='best')

os.makedirs(f"{out_dir}/all_samples", exist_ok=True)

fig.savefig(f"{out_dir}/all_samples/libraries.png", dpi=300)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 6))
sc.pl.umap(adata, color='log1p_total_counts', show=False, ax=ax)

fig.savefig(f"{out_dir}/all_samples/log1p_total_counts.png", dpi=300)
plt.close(fig)

In [ ]:
for topic in topic_prop.columns.tolist():
    fig, ax = plt.subplots(figsize=(6, 5))
    sc.pl.umap(adata, color=topic, show=False, ax=ax)

    fig.savefig(f"{out_dir}/all_samples/{topic}.png", dpi=300)
    plt.close(fig)

In [ ]:
# visualize topic proportions on spatial coordinates
for sample in sample_list:
    os.makedirs(f"{out_dir}/{sample}", exist_ok=True)

    df_obs = adata[adata.obs['library_id'] == sample].obs
    topic_prop = df_obs.loc[:, df_obs.columns.str.startswith("Topic")]

    adata_sub = sc.read_h5ad(f"{in_dir}/{sample}.h5ad")

    adata_sub.obs = adata_sub.obs.join(topic_prop, how='inner')
    
    for topic in topic_prop.columns.tolist():
        fig, ax = plt.subplots(figsize=(6, 6))
        sc.pl.spatial(adata_sub, color=topic, show=False, ax=ax)

        fig.savefig(f"{out_dir}/{sample}/{topic}.png", dpi=300)
        plt.close(fig)

In [ ]:
adata.write_h5ad(f"{out_dir}/scTM_15topics.h5ad")